# Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from constants import Constant as C
from loaders import load_ratings
from models import ContentBased
from python_helper import submit_predictions
# `submit_predictions` auto-loads .env from the repo root on import,
# so HACKATHON_URL and HACKATHON_TOKEN become available without any
# manual os.environ setup.

# How to submit predictions to the hackathon

## One-time setup: create a `.env` file at the repo root

Add the URL and your group's token (provided by the instructor):

```env
HACKATHON_URL=https://recsys-hackathon.vercel.app
HACKATHON_TOKEN=your-group-token-here
```

`.env` is gitignored. If you change it, restart the Jupyter kernel to pick up the new values (the helper loads `.env` once at import time).

## Workflow

1. **Tune offline.** Use your evaluator notebook to pick the best `feature_method` and `regressor_method` for `ContentBased`. Every submission to the server counts toward your quota — iterate locally before submitting.
2. **Predict.** Run `make_hackathon_prediction(...)` below to produce `df_predictions`.
3. **Submit.** Run `submit_predictions(df_predictions)` — the server scores it and prints your RMSE, rank, and remaining quota.

## Quota per group

`10s` cooldown between submissions · `10` per hour · `300` total over the hackathon. **Errors count toward the quota**, so verify the CSV format works once before iterating in a tight loop.

In [ ]:
def make_hackathon_prediction(feature_method, regressor_method):
    """Train a ContentBased model and produce predictions on the test set.

    Returns a DataFrame with columns ['userId', 'movieId', 'rating'] in the
    exact order of the hidden test set. Pass it to submit_predictions().
    """
    # 1) load train data - make sure to redirect the DATA_PATH to 'data/hackathon'
    assert str(C.DATA_PATH) == 'data/hackathon'
    sp_ratings = load_ratings(surprise_format=True)
    train_set = sp_ratings.build_full_trainset()

    # 2) train your ContentBased model on the train set
    content_knn = ContentBased(feature_method, regressor_method)
    content_knn.fit(train_set)

    # 3) make predictions on the test set
    df_test = pd.read_csv('data/hackathon/evidence/ratings_test.csv')[C.USER_ITEM_RATINGS]
    test_records = list(df_test.to_records(index=False))
    predictions = content_knn.test(test_records)
    output_predictions = []
    for uid, iid, _, est, _ in predictions:
        output_predictions.append([uid, iid, est])
    df_predictions = pd.DataFrame(data=output_predictions, columns=df_test.columns)

    return df_predictions


df_predictions = make_hackathon_prediction('title_length', 'linear')
df_predictions.head()

In [ ]:
submit_predictions(df_predictions)

In [ ]:
# Optional: peek at your remaining quota without burning a submission
from python_helper import check_quota
check_quota()